This is the code to generate the plot of cluster overlay for WSIs. And we provide various plot including high-attention map, plain concept map, concept map with attention added as alpha score

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
WSI cluster overlays (coords = TOP-LEFT @ level-0) with:
  • Per-mode cluster→color remapping (RAW vs RAWH)
  • One-line legend centered below the image (all overlays except ATTN_DENSITY)
  • Two-pass rendering: fast (small) then hires (from cache)
  • Smoke test modes for 3 slides
  • Distance- and attention-aware alphas
  • NEW MODES:
        RAW               : RAW clusters (distance alpha)
        RAWH_DIST         : h-space clusters (distance alpha)
        RAWH_ATTNFUSE     : h-space clusters with alpha = attn^γa * conf^γc
        RAWH_TOPQ         : h-space clusters, draw only top-q% attention tiles
        ATTN_DENSITY      : attention-only density heatmap (no legend)
  • Output files per slide/mode (except ATTN_DENSITY which has no legend):
        <slide>_<mode>_ORIGINAL.png
        <slide>_<mode>_COMPOSITE_WITH_LEGEND.png
        <slide>_<mode>_OVERLAY_ONLY_WITH_LEGEND.png
"""

from pathlib import Path
import gc, h5py, joblib, warnings
import numpy as np, pandas as pd, torch, openslide
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import matplotlib; matplotlib.use("Agg")
import matplotlib
import matplotlib.colors as mcolors
import matplotlib.cm as cm

# -------------------- PATHS --------------------
SVS_ROOT     = Path("/common/users/wq50/CLAM/HNSCC_slides")
FEAT_DIR     = Path("/common/users/wq50/CLAM/features/HPV_UNI2_features/h5_files")
RAW_MODEL    = Path("/common/users/wq50/CLAM2/kmeans_models/hpv_uni2_k10_raw.joblib")
RAWH_MODEL   = Path("/common/users/wq50/CLAM2/kmeans_models/hpv_uni2_k10_rawh.joblib")
CLAM_WEIGHT  = Path("/common/users/wq50/CLAM/results/HPV_CLAM_50_mb_s1/s_9_checkpoint.pt")
EMBED_DIM    = 1536
LABELS_CSV   = Path("/common/users/wq50/CLAM/dataset_csv/HNSCC.csv")

OUT_DIR      = Path("overlay_plots_mapped"); OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR    = Path("cache_npz"); CACHE_DIR.mkdir(parents=True, exist_ok=True)

# -------------------- SETTINGS --------------------
DEVICE            = "cuda" if torch.cuda.is_available() else "cpu"
BATCH             = 16_384
K_ASSERT          = 10
TILE_SIZE_L0      = 256

# Run modes:
#   "fast"         -> compute+cache+render ALL slides at THUMB_MAX_W_FAST
#   "hires"        -> re-render from cache for ALL slides at THUMB_MAX_W_HIRES
#   "smoke3_fast"  -> fast for 3 slides
#   "smoke3_hires" -> hires redraw for the same 3 slides
RUN_MODE          = "fast"

THUMB_MAX_W_FAST  = 1200
THUMB_MAX_W_HIRES = 4000
SMOKE3_RANDOMIZE  = False

# Alpha for distance-only modes
ALPHA_MODE   = "rank"          # 'rank' | 'percentile' | 'sigmoid'
ALPHA_Q      = 0.98            # for 'percentile'
ALPHA_GAMMA  = 1.0
SIGMOID_K    = 8.0
MIN_ALPHA    = 80              # 0..255
DRAW_BORDER  = False
BORDER_ALPHA = 0               # ignored when DRAW_BORDER=False

# Attention-aware parameters (RAWH_ATTNFUSE / RAWH_TOPQ / ATTN_DENSITY)
ATTN_GAMMA   = 1.3     # attention exponent
CONF_GAMMA   = 1.0     # distance-confidence exponent
BASE_ALPHA   = 0.90    # alpha cap in [0..1]
TOPQ         = 0.20    # top-20% attention for RAWH_TOPQ (set None to disable)

# ATTN_DENSITY rendering
DENSITY_BLUR_SIGMA = 2.0        # post Gaussian blur (thumbnail space)
DENSITY_CMAP       = "magma"    # perceptual colormap
DENSITY_ALPHA      = 0.70       # overlay alpha for density on slide

SVS_EXTS     = {".svs", ".tif", ".tiff", ".ndpi", ".mrxs"}

# -------------------- CLUSTER→COLOR REMAPPING --------------------
RAW_COLOR_MAP  = {}  # e.g., {1: 3}
RAWH_COLOR_MAP = {}  # e.g., {1: 0}

# -------------------- CLAM (h-space) --------------------
from models.model_clam import CLAM_MB

@torch.inference_mode()
def load_clam(weight_path: Path, device: str, embed_dim: int):
    m = CLAM_MB(gate=True, size_arg="small", n_classes=2, embed_dim=embed_dim)
    sd = torch.load(weight_path, map_location=device)
    m.load_state_dict(sd, strict=False)
    return m.to(device).eval()

@torch.inference_mode()
def project_h_and_attention(block_np: np.ndarray, clam: CLAM_MB, device: str):
    x = torch.from_numpy(block_np).to(device)
    A_raw, h = clam.attention_net(x)  # A_raw: (m,2)
    logits = A_raw[:, 0]
    a = torch.softmax(logits, dim=0).clamp_min(1e-12)
    # renorm so mean ≈ 1
    a = a * (a.numel() / a.sum())
    return h.cpu().numpy().astype(np.float32), a.cpu().numpy().astype(np.float32)

# -------------------- IO helpers --------------------
def iter_h5(h5_path: Path, batch=BATCH):
    with h5py.File(h5_path, "r") as f:
        X = f["features"]; C = f["coords"]  # coords: TOP-LEFT @ LEVEL-0
        N = X.shape[0]
        for off in range(0, N, batch):
            yield off, X[off:off+batch][:].astype(np.float32), C[off:off+batch][:].astype(np.int32)

def open_thumbnail(svs_path: Path, max_w: int):
    slide = openslide.OpenSlide(str(svs_path))
    W, H = slide.dimensions
    scale = max_w / W if W > max_w else 1.0
    thumb = slide.get_thumbnail((int(W*scale), int(H*scale))).convert("RGB")
    slide.close()
    return thumb, scale

# -------------------- Palettes --------------------
def base_palette_colors(K: int):
    base_hex = [
        "#E41A1C",  # red
        "#377EB8",  # blue
        "#031B2E",  # green
        "#984EA3",  # purple
        "#FF7F00",  # orange
        "#FFD92F",  # yellow
        "#F781BF",  # pink
        "#66C2A5",  # teal
        "#A65628",  # brown
        "#999999",  # gray
    ]
    base = np.array([mcolors.to_rgb(h) for h in base_hex])
    reps = int(np.ceil(K / len(base)))
    return (np.tile(base, (reps, 1))[:K] * 255).astype(np.uint8)

def build_mode_colors(K: int, mapping: dict[int, int]):
    base = base_palette_colors(K)
    out = np.zeros_like(base)
    for k in range(K):
        src = mapping.get(k, k)
        src = int(np.clip(src, 0, K-1))
        out[k] = base[src]
    return out  # uint8 (K,3)

# -------------------- Assign + distance --------------------
def kmeans_assign_and_dist(X: np.ndarray, km) -> tuple[np.ndarray, np.ndarray]:
    labs = km.predict(X)
    C = km.cluster_centers_.astype(np.float32)
    d = np.linalg.norm(X - C[labs], axis=1)
    return labs.astype(np.int32), d.astype(np.float32)

# -------------------- Alpha (distance-only) --------------------
def alphas_percentile(labels, dists, K, q=ALPHA_Q, gamma=ALPHA_GAMMA, min_alpha=MIN_ALPHA):
    qk = np.zeros(K, dtype=np.float32)
    for k in range(K):
        dk = dists[labels==k]
        qk[k] = np.quantile(dk, q) if dk.size else 1.0
        if qk[k] <= 1e-9: qk[k] = 1.0
    a = 1.0 - (dists / qk[labels])
    a = np.clip(a, 0.0, 1.0) ** gamma
    a = np.maximum(a, min_alpha/255.0)
    return a

def alphas_rank(labels, dists, K, gamma=ALPHA_GAMMA, min_alpha=MIN_ALPHA):
    a = np.zeros_like(dists, dtype=np.float32)
    for k in range(K):
        idx = np.where(labels==k)[0]
        if idx.size == 0: continue
        dk = dists[idx]
        ranks = np.argsort(np.argsort(dk))
        frac  = 1.0 - ranks.astype(np.float32) / max(1, idx.size-1)
        frac  = np.clip(frac, 0.0, 1.0) ** gamma
        a[idx] = frac
    a = np.maximum(a, min_alpha/255.0)
    return a

def alphas_sigmoid(labels, dists, K, k=SIGMOID_K, min_alpha=MIN_ALPHA):
    a = np.zeros_like(dists, dtype=np.float32)
    for c in range(K):
        idx = np.where(labels==c)[0]
        if idx.size == 0: continue
        dk = dists[idx]
        med = np.median(dk)
        q75 = np.quantile(dk, 0.75); q25 = np.quantile(dk, 0.25)
        scale = max(q75 - q25, 1e-6)
        z = (med - dk) / scale
        sig = 1.0 / (1.0 + np.exp(-k * z))
        a[idx] = sig
    a = np.maximum(a, min_alpha/255.0)
    return a

def compute_alpha_dist(labels, dists, K):
    if ALPHA_MODE == "percentile":
        return alphas_percentile(labels, dists, K)
    elif ALPHA_MODE == "rank":
        return alphas_rank(labels, dists, K)
    elif ALPHA_MODE == "sigmoid":
        return alphas_sigmoid(labels, dists, K)
    else:
        raise ValueError(f"Unknown ALPHA_MODE: {ALPHA_MODE}")

# -------------------- Attention fusion helpers --------------------
def _norm01(x, eps=1e-12):
    x = np.asarray(x, np.float64)
    lo, hi = np.nanmin(x), np.nanmax(x)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi - lo < eps:
        return np.zeros_like(x, dtype=np.float64)
    return (x - lo) / max(hi - lo, eps)

def dist_confidence(H, centers, labels):
    C = centers[labels]
    d = np.sqrt(((H - C) ** 2).sum(1))
    conf = np.zeros_like(d, dtype=np.float64)
    for k in range(centers.shape[0]):
        idx = (labels == k)
        if not np.any(idx): continue
        med = np.median(d[idx])
        if med <= 0: conf[idx] = 1.0
        else:        conf[idx] = 1.0 - d[idx] / (med + 1e-9)
    return _norm01(conf)

def alpha_fusion(attn01, conf01, attn_gamma=ATTN_GAMMA, conf_gamma=CONF_GAMMA, base_alpha=BASE_ALPHA, min_alpha=MIN_ALPHA):
    a = np.power(np.clip(attn01, 0, 1), attn_gamma)
    c = np.power(np.clip(conf01, 0, 1), conf_gamma)
    out = np.clip(a * c, 0.0, base_alpha)
    out = np.maximum(out, min_alpha/255.0)
    return out

# -------------------- Legend (single row, centered) --------------------
def make_bottom_legend(colors: np.ndarray, width: int, pad: int = 20, sw: int = 22, text_gap: int = 6):
    K = colors.shape[0]
    try:
        font = ImageFont.load_default()
        bbox = font.getbbox("C0"); text_h = bbox[3]
    except Exception:
        font = None; text_h = 10

    legend_h = pad + sw + text_gap + text_h + pad
    legend = Image.new("RGB", (width, legend_h), (255, 255, 255))
    draw = ImageDraw.Draw(legend)

    xs = np.linspace(pad, width - pad, K) if K > 1 else np.array([width // 2])
    y0 = pad
    for i, xc in enumerate(xs):
        xc = int(round(xc))
        x1 = xc - sw // 2; y1 = y0
        x2 = x1 + sw;      y2 = y1 + sw
        color = tuple(map(int, colors[i]))
        draw.rectangle([x1, y1, x2, y2], fill=color, outline=(0, 0, 0))
        label = f"C{i}"
        if font:
            tw, th = font.getbbox(label)[2], font.getbbox(label)[3]
        else:
            tw, th = int(len(label)*6), text_h
        tx = int(round(xc - tw / 2))
        ty = y2 + text_gap
        draw.text((tx, ty), label, fill=(0, 0, 0), font=font)
    return legend

def stack_image_with_legend(img: Image.Image, legend: Image.Image, pad: int = 10):
    W = img.width
    if legend.width != W:
        legend = legend.resize((W, legend.height), Image.BILINEAR)
    canvas = Image.new("RGB", (W, img.height + pad + legend.height), (255, 255, 255))
    canvas.paste(img.convert("RGB"), (0, 0))
    canvas.paste(legend, (0, img.height + pad))
    return canvas

# -------------------- Drawing --------------------
def draw_overlay(thumb: Image.Image,
                 coords_l0: np.ndarray,
                 labels: np.ndarray,
                 dists: np.ndarray,
                 colors: np.ndarray,
                 scale: float,
                 tile_size_l0: int = TILE_SIZE_L0,
                 alpha_vec: np.ndarray | None = None) -> Image.Image:
    Wt, Ht = thumb.size
    overlay = Image.new("RGBA", (Wt, Ht), (0,0,0,0))
    draw = ImageDraw.Draw(overlay, "RGBA")

    K = colors.shape[0]
    alpha = compute_alpha_dist(labels, dists, K) if alpha_vec is None else alpha_vec
    w = int(tile_size_l0 * scale)
    border_w = max(1, w // 18) if DRAW_BORDER else 0

    for (x0, y0), lab, a in zip(coords_l0, labels, alpha):
        x = int(x0 * scale); y = int(y0 * scale)
        rgba = tuple(map(int, colors[lab])) + (int(255*a),)
        draw.rectangle([x, y, x+w, y+w], fill=rgba, outline=None)
        if DRAW_BORDER:
            brgba = (0, 0, 0, BORDER_ALPHA)
            draw.rectangle([x, y, x+w, y+w], outline=brgba, width=border_w)

    return overlay

def overlay_on_white(overlay_rgba: Image.Image) -> Image.Image:
    bg = Image.new("RGB", overlay_rgba.size, (255,255,255))
    return Image.alpha_composite(bg.convert("RGBA"), overlay_rgba).convert("RGB")

def save_with_legend_only(slide_id: str, mode: str, thumb: Image.Image, overlay: Image.Image, colors: np.ndarray, out_dir: Path):
    # Original
    p_orig = out_dir / f"{slide_id}_{mode}_ORIGINAL.png"
    thumb.convert("RGB").save(p_orig, quality=95)

    # Composite + legend
    comp = Image.alpha_composite(thumb.convert("RGBA"), overlay).convert("RGB")
    legend = make_bottom_legend(colors, width=comp.width)
    # comp_L = stack_image_with_legend(comp, legend)
    comp_L = comp
    p_compL = out_dir / f"{slide_id}_{mode}_COMPOSITE_WITH_LEGEND.png"
    comp_L.save(p_compL, quality=95)

    # Overlay-only on white + legend
    over_white = overlay_on_white(overlay)
    # over_L = stack_image_with_legend(over_white, legend)
    over_L = over_white
    p_overL = out_dir / f"{slide_id}_{mode}_OVERLAY_ONLY_WITH_LEGEND.png"
    over_L.save(p_overL, quality=95)

    return {
        "original_png": str(p_orig),
        "composite_with_legend_png": str(p_compL),
        "overlay_only_with_legend_png": str(p_overL),
    }

def save_density(slide_id: str, thumb: Image.Image, density_img: Image.Image, out_dir: Path):
    # heatmap only
    p_heat = out_dir / f"{slide_id}_ATTN_DENSITY_ONLY.png"
    density_img.save(p_heat, quality=95)
    # overlay on slide
    comp = Image.blend(thumb.convert("RGB"), density_img.convert("RGB"), alpha=DENSITY_ALPHA)
    p_comp = out_dir / f"{slide_id}_ATTN_DENSITY_ON_SLIDE.png"
    comp.save(p_comp, quality=95)
    return {"density_only": str(p_heat), "density_on_slide": str(p_comp)}

# -------------------- Slide helpers --------------------
def collect_slide_stems(root: Path) -> set[str]:
    stems = set()
    for p in root.rglob("*"):
        if p.suffix.lower() in SVS_EXTS:
            stems.add(p.stem)
    return stems

def find_svs_by_stem(root: Path, stem: str) -> Path | None:
    for p in root.rglob("*"):
        if p.suffix.lower() in SVS_EXTS and p.stem == stem:
            return p
    return None

# -------------------- Cache helpers --------------------
# For modes that need attention, we store 'attn'. For RAW it remains empty.
def save_cache(slide_id: str, mode: str, labels_all, dists_all, coords_all, attn_all=None):
    if attn_all is None:
        np.savez_compressed(
            CACHE_DIR / f"{slide_id}_{mode}.npz",
            labels=labels_all.astype(np.int32),
            dists=dists_all.astype(np.float32),
            coords=coords_all.astype(np.int32),
        )
    else:
        np.savez_compressed(
            CACHE_DIR / f"{slide_id}_{mode}.npz",
            labels=labels_all.astype(np.int32),
            dists=dists_all.astype(np.float32),
            coords=coords_all.astype(np.int32),
            attn=attn_all.astype(np.float32),
        )

def load_cache(slide_id: str, mode: str):
    p = CACHE_DIR / f"{slide_id}_{mode}.npz"
    if not p.exists():
        return None
    dat = np.load(p)
    labels = dat["labels"]; dists = dat["dists"]; coords = dat["coords"]
    attn = dat["attn"] if "attn" in dat.files else None
    return labels, dists, coords, attn

# -------------------- Rendering pipelines --------------------
def render_compute_and_save_RAW(slide_id, svs_path, km_raw, colors, thumb_w):
    h5 = FEAT_DIR / f"{slide_id}.h5"
    if not h5.exists(): return None

    labels_all, dists_all, coords_all = [], [], []
    for _, blk, crd in iter_h5(h5, BATCH):
        labs, d = kmeans_assign_and_dist(blk, km_raw)
        labels_all.append(labs); dists_all.append(d); coords_all.append(crd)
        del labs, d, blk; gc.collect()

    labels_all = np.concatenate(labels_all); dists_all = np.concatenate(dists_all)
    coords_all = np.concatenate(coords_all)
    save_cache(slide_id, "RAW", labels_all, dists_all, coords_all, attn_all=None)

    thumb, scale = open_thumbnail(svs_path, thumb_w)
    overlay = draw_overlay(thumb, coords_all, labels_all, dists_all, colors, scale, tile_size_l0=TILE_SIZE_L0)
    return save_with_legend_only(slide_id, "RAW", thumb, overlay, colors, OUT_DIR)

def render_compute_and_save_RAWH_variants(slide_id, svs_path, km_rawh, clam, colors, thumb_w):
    """
    Computes h, attention, labels, dists once; saves three RAWH figures:
      - RAWH_DIST (distance alpha)
      - RAWH_ATTNFUSE (alpha = attn×conf)
      - RAWH_TOPQ (top-q% attention only)
    Also saves ATTN_DENSITY heatmap images.
    """
    h5 = FEAT_DIR / f"{slide_id}.h5"
    if not h5.exists(): return None

    labels_all, dists_all, coords_all, attn_all, h_all = [], [], [], [], []
    for _, blk, crd in iter_h5(h5, BATCH):
        H, a = project_h_and_attention(blk, clam, DEVICE)
        labs, d = kmeans_assign_and_dist(H, km_rawh)
        labels_all.append(labs); dists_all.append(d); coords_all.append(crd); attn_all.append(a); h_all.append(H)
        del labs, d, blk, H, a; gc.collect()

    labels_all = np.concatenate(labels_all)
    dists_all  = np.concatenate(dists_all)
    coords_all = np.concatenate(coords_all)
    attn_all   = np.concatenate(attn_all)
    H_concat   = np.concatenate(h_all).astype(np.float32)

    # normalize attention to 0..1 (across slide)
    attn01 = _norm01(attn_all)
    save_cache(slide_id, "RAWH", labels_all, dists_all, coords_all, attn_all=attn01)

    thumb, scale = open_thumbnail(svs_path, thumb_w)

    # (1) RAWH_DIST — distance alpha only (as before)
    overlay_dist = draw_overlay(thumb, coords_all, labels_all, dists_all, colors, scale, tile_size_l0=TILE_SIZE_L0)
    save_with_legend_only(slide_id, "RAWH_DIST", thumb, overlay_dist, colors, OUT_DIR)

    # (2) RAWH_ATTNFUSE — alpha = attention × distance confidence
    conf01 = dist_confidence(H_concat, km_rawh.cluster_centers_.astype(np.float32), labels_all)
    alpha_fused = alpha_fusion(attn01, conf01, ATTN_GAMMA, CONF_GAMMA, BASE_ALPHA, MIN_ALPHA)
    overlay_fuse = draw_overlay(thumb, coords_all, labels_all, dists_all, colors, scale, tile_size_l0=TILE_SIZE_L0, alpha_vec=alpha_fused)
    save_with_legend_only(slide_id, "RAWH_ATTNFUSE", thumb, overlay_fuse, colors, OUT_DIR)

    # (3) RAWH_TOPQ — draw only top-q% attention tiles
    if TOPQ is not None:
        thr = np.quantile(attn01, 1.0 - TOPQ)
        keep = attn01 >= thr
        # Use a high opaque alpha for kept points, else skip
        alpha_topq = np.zeros_like(attn01, dtype=np.float32)
        alpha_topq[keep] = np.maximum(BASE_ALPHA, MIN_ALPHA/255.0)
        overlay_topq = draw_overlay(thumb, coords_all, labels_all, dists_all, colors, scale, tile_size_l0=TILE_SIZE_L0, alpha_vec=alpha_topq)
        save_with_legend_only(slide_id, f"RAWH_TOP{int(TOPQ*100)}", thumb, overlay_topq, colors, OUT_DIR)

    # (4) ATTN_DENSITY — attention-only heatmap (no legend)
    Wt, Ht = thumb.size
    dens = np.zeros((Ht, Wt), dtype=np.float32)
    w = int(TILE_SIZE_L0 * scale)
    xs = (coords_all[:,0] * scale).astype(int)
    ys = (coords_all[:,1] * scale).astype(int)
    # accumulate attention mass per tile footprint (simple box splat)
    for x, y, a in zip(xs, ys, attn01):
        x2 = min(Wt, x + w); y2 = min(Ht, y + w)
        dens[y:y2, x:x2] += float(a)
    # blur for nicer density
    dens_img = Image.fromarray((_norm01(dens)*255).astype(np.uint8))
    dens_img = dens_img.filter(ImageFilter.GaussianBlur(radius=DENSITY_BLUR_SIGMA))
    # apply colormap
    cmap = matplotlib.colormaps.get_cmap(DENSITY_CMAP)
    dens_rgb = (np.array(cmap(np.array(dens_img)/255.0))[:,:,:3]*255).astype(np.uint8)
    dens_pil = Image.fromarray(dens_rgb, mode="RGB")
    save_density(slide_id, thumb, dens_pil, OUT_DIR)

    return True

def render_from_cache_and_save(slide_id: str, svs_path: Path, mode: str, colors: np.ndarray, thumb_w: int):
    cached = load_cache(slide_id, mode)
    if cached is None:
        print(f"[skip] cache missing for {slide_id} {mode}")
        return None
    labels_all, dists_all, coords_all, attn = cached
    thumb, scale = open_thumbnail(svs_path, thumb_w)

    if mode == "RAW":
        overlay = draw_overlay(thumb, coords_all, labels_all, dists_all, colors, scale, tile_size_l0=TILE_SIZE_L0)
        return save_with_legend_only(slide_id, mode, thumb, overlay, colors, OUT_DIR)

    # RAWH cached: labels/dists/coords + attn
    # Re-render all RAWH variants + density from cache
    # (distance-only)
    overlay_dist = draw_overlay(thumb, coords_all, labels_all, dists_all, colors, scale, tile_size_l0=TILE_SIZE_L0)
    save_with_legend_only(slide_id, "RAWH_DIST", thumb, overlay_dist, colors, OUT_DIR)

    # fused alpha = attn × (proxy confidence via per-cluster percentile dists)
    # Use per-cluster normalized 1 - d/quantile for a proxy of confidence
    K = colors.shape[0]
    # quick confidence proxy from dists only (if H not cached): percentile per cluster
    qk = np.zeros(K, dtype=np.float32)
    for k in range(K):
        dk = dists_all[labels_all==k]
        qk[k] = np.quantile(dk, 0.5) if dk.size else 1.0
        if qk[k] <= 1e-9: qk[k] = 1.0
    conf01 = 1.0 - (dists_all / qk[labels_all])
    conf01 = _norm01(np.clip(conf01, 0.0, 1.0))
    attn01 = _norm01(attn) if attn is not None else np.zeros_like(conf01)
    alpha_fused = alpha_fusion(attn01, conf01, ATTN_GAMMA, CONF_GAMMA, BASE_ALPHA, MIN_ALPHA)
    overlay_fuse = draw_overlay(thumb, coords_all, labels_all, dists_all, colors, scale, tile_size_l0=TILE_SIZE_L0, alpha_vec=alpha_fused)
    save_with_legend_only(slide_id, "RAWH_ATTNFUSE", thumb, overlay_fuse, colors, OUT_DIR)

    if TOPQ is not None and attn is not None:
        thr = np.quantile(attn01, 1.0 - TOPQ)
        keep = attn01 >= thr
        alpha_topq = np.zeros_like(attn01, dtype=np.float32)
        alpha_topq[keep] = np.maximum(BASE_ALPHA, MIN_ALPHA/255.0)
        overlay_topq = draw_overlay(thumb, coords_all, labels_all, dists_all, colors, scale, tile_size_l0=TILE_SIZE_L0, alpha_vec=alpha_topq)
        save_with_legend_only(slide_id, f"RAWH_TOP{int(TOPQ*100)}", thumb, overlay_topq, colors, OUT_DIR)

    # Attention density from attn cache
    Wt, Ht = thumb.size
    dens = np.zeros((Ht, Wt), dtype=np.float32)
    w = int(TILE_SIZE_L0 * scale)
    xs = (coords_all[:,0] * scale).astype(int)
    ys = (coords_all[:,1] * scale).astype(int)
    a01 = attn01
    for x, y, a in zip(xs, ys, a01):
        x2 = min(Wt, x + w); y2 = min(Ht, y + w)
        dens[y:y2, x:x2] += float(a)
    dens_img = Image.fromarray((_norm01(dens)*255).astype(np.uint8))
    dens_img = dens_img.filter(ImageFilter.GaussianBlur(radius=DENSITY_BLUR_SIGMA))
    cmap = matplotlib.colormaps.get_cmap(DENSITY_CMAP)
    dens_rgb = (np.array(cmap(np.array(dens_img)/255.0))[:,:,:3]*255).astype(np.uint8)
    dens_pil = Image.fromarray(dens_rgb, mode="RGB")
    save_density(slide_id, thumb, dens_pil, OUT_DIR)
    return True

# -------------------- Main --------------------
def main():
    df_lab = pd.read_csv(LABELS_CSV)
    assert "slide_id" in df_lab.columns, "LABELS_CSV must include 'slide_id'."
    csv_ids = set(df_lab["slide_id"].astype(str))

    stems = collect_slide_stems(SVS_ROOT)
    assert csv_ids == stems, "Slide ID ↔ filename mismatch (by stem)."

    feat_slides = {p.stem for p in FEAT_DIR.glob("*.h5")}
    target_all = sorted(csv_ids & feat_slides)
    print(f"[discover] {len(target_all)} slides with features+SVS.")

    # --- SMOKE3 selection ---
    if RUN_MODE in ("smoke3_fast", "smoke3_hires"):
        if SMOKE3_RANDOMIZE:
            rng = np.random.default_rng(0)
            target = list(rng.choice(target_all, size=min(3, len(target_all)), replace=False))
        else:
            target = target_all[:3]   # deterministic first 3
        print(f"[smoke3] using {len(target)} slides: {target}")
    else:
        target = target_all

    km_raw  = joblib.load(RAW_MODEL)
    km_rawh = joblib.load(RAWH_MODEL)
    assert km_raw.n_clusters == K_ASSERT and km_rawh.n_clusters == K_ASSERT, "Models must be k=10."

    colors_raw  = build_mode_colors(km_raw.n_clusters,  RAW_COLOR_MAP)
    colors_rawh = build_mode_colors(km_rawh.n_clusters, RAWH_COLOR_MAP)

    if RUN_MODE in ("fast", "smoke3_fast"):
        clam = load_clam(CLAM_WEIGHT, DEVICE, EMBED_DIM)
        for sid in target:
            svs_path = find_svs_by_stem(SVS_ROOT, sid)
            if svs_path is None:
                print(f"[skip] SVS not found for {sid}")
                continue
            # RAW baseline
            render_compute_and_save_RAW(sid, svs_path, km_raw, colors_raw, THUMB_MAX_W_FAST)
            # RAWH variants + attention density
            render_compute_and_save_RAWH_variants(sid, svs_path, km_rawh, clam, colors_rawh, THUMB_MAX_W_FAST)

    elif RUN_MODE in ("hires", "smoke3_hires"):
        for sid in target:
            svs_path = find_svs_by_stem(SVS_ROOT, sid)
            if svs_path is None:
                print(f"[skip] SVS not found for {sid}")
                continue
            render_from_cache_and_save(sid, svs_path, "RAW",  colors_raw,  THUMB_MAX_W_HIRES)
            render_from_cache_and_save(sid, svs_path, "RAWH", colors_rawh, THUMB_MAX_W_HIRES)

    else:
        raise ValueError("RUN_MODE must be one of: 'fast', 'hires', 'smoke3_fast', 'smoke3_hires'")

    print(f"Done. RUN_MODE={RUN_MODE} | slides={len(target)} | "
          f"ALPHA_MODE={ALPHA_MODE} | MIN_ALPHA={MIN_ALPHA} | "
          f"ATTN_GAMMA={ATTN_GAMMA} | CONF_GAMMA={CONF_GAMMA} | TOPQ={TOPQ}")

if __name__ == "__main__":
    warnings.filterwarnings("ignore", message=".*get_cmap function was deprecated.*")
    torch.set_float32_matmul_precision("high")
    np.random.seed(0); torch.manual_seed(0)
    main()